In [ ]:
TICKERS = [
    'ACM', 'ADEA', 'AMKR', 'AMZN','AVDV', 'AVGO',
    'COPJ', 
    "DAVE",'DRNZ', 
    'ENS',
    'FPS',  
    'GEV', 'GLD', 'GLW', 'GOOG',
    'HWM',  
    'IVV',  
    'KOID',
    'LDOS', 
    'MELI', 'MLI', 
    'NBIS',
    'REMX', 
    'UMAC',
    'VDIGX','VEXC', 'VOOG', 'VTI',  'VXUS', 'VICR',

     "XLI", "XLP", "XLY", "XLE", "XLRE", "XLU", "XLB", "XLC", 'XLF', 'SPY', 'QQQ',
]
]

OUTPUT_CSV  = r"G:\My Drive\Projects\Momentum\Momentum.csv"
OUTPUT_HTML = r"C:\Users\hayes\OneDrive\Desktop\Momentum.dashboard.html"


In [22]:
#@title Prepare the Momentum CSV and DF
# !/usr/bin/env python3
# =============================================================================
#  momentum_squeeze_with_sells.py
#  Standalone momentum + TTM-squeeze + sell-signal indicator builder
# -----------------------------------------------------------------------------
#  FINAL VERSION — includes:
#    • 7-factor momentum composite (benchmark-anchored vs SPY)
#    • TTM Squeeze family + setup stack
#    • Sell signals: RSI ≥ 70/80, PPO signal-line cross, PPO zero-line cross,
#      distance (PPO − Signal), price vs EMA20/50/200
# =============================================================================

import numpy as np
if not hasattr(np, "NaN"):
    np.NaN = np.nan
if not hasattr(np, "Inf"):
    np.Inf = np.inf

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import pandas_ta as ta
import yfinance as yf

# =============================================================================
#  CONFIG
# =============================================================================

PERIOD = "2y"
BENCHMARK = "SPY"
BENCH_PERIOD = "5y"

MOM_LOOKBACK = 252
MOM_SKIP = 21
HIGH_WINDOW = 252
HIGH_MINP = 126
MIN_BARS = MOM_LOOKBACK + MOM_SKIP
TRIM_BARS = 320
LIQ_WINDOW = 20
OBV_WINDOW = 20
SQZ_RECENT_WINDOW = 10

SQZ = dict(bb_length=20, bb_std=2.0, kc_length=20, kc_scalar=1.5, mom=12, use_tr=True)

MOM_TRAILING_WEIGHTS = {
    "Momentum_12_1": 0.45,
    "ADX_Trend": 0.30,
    "High_52W_Ratio": 0.25,
}
MOM_EMERGING_WEIGHTS = {
    "PPO_Delta": 0.25,
    "DMI_Value": 0.25,
    "OBV_slope": 0.20,
    "Squeeze_Mom_Pct": 0.20,
    "Trend_Count": 0.10,
}
MOM_TRAILING_WT = 0.60
MOM_EMERGING_WT = 0.40

MIN_DOLLAR_VOL = 5_000_000
MIN_PRICE = 5.00
MIN_COMPRESSION = 15
DMI_FLOOR = -5
RSI_LOW, RSI_HIGH = 45, 65
ROOM_TO_RUN_MAX = 0.90
FLAT_MOM_LOW, FLAT_MOM_HIGH = 20, 60

RSI_OVERBOUGHT = 70
RSI_EXTREME = 80
EMA_FAST = 20
EMA_MID = 50
EMA_SLOW = 200
SELL_ALERT_THRESHOLD = 3


BENCH_COMPONENTS = list(MOM_TRAILING_WEIGHTS) + list(MOM_EMERGING_WEIGHTS)


# =============================================================================
#  1.  PRICE DATA
# =============================================================================
def _tidy(df):
    return (df.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]
              .dropna(how="all"))


def fetch_prices(tickers, period=PERIOD):
    raw = yf.download(tickers, period=period, interval="1d", auto_adjust=True,
                      group_by="ticker", progress=False, threads=True)
    out = {}
    for t in tickers:
        try:
            df = _tidy(raw[t] if isinstance(raw.columns, pd.MultiIndex) else raw)
            if len(df) >= MIN_BARS // 2:
                out[t] = df
            else:
                print(f"  WARNING: {t} has only {len(df)} bars — skipped")
        except Exception as e:
            print(f"  WARNING: could not load {t} ({e}) — skipped")
    return out


def fetch_one(ticker, period):
    raw = yf.download(ticker, period=period, interval="1d", auto_adjust=True,
                      progress=False, threads=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    return _tidy(raw)


# =============================================================================
#  2.  RAW INDICATORS + SELL SIGNALS
# =============================================================================
def _obv_slope(df):
    obv = ta.obv(close=df["close"], volume=df["volume"])
    scale = df["volume"].rolling(OBV_WINDOW, min_periods=OBV_WINDOW).mean().replace(0, np.nan)
    norm = obv / scale

    def _beta(w):
        if np.isnan(w).any():
            return np.nan
        t = np.arange(len(w))
        return np.polyfit(t, w, 1)[0]

    return norm.rolling(OBV_WINDOW, min_periods=OBV_WINDOW).apply(_beta, raw=True)


def _trend_count_and_cross(df):
    """PPO, signal, histogram, trend count, signal-line & zero-line crosses."""
    ema12 = ta.ema(df["close"], length=12)
    ema26 = ta.ema(df["close"], length=26)
    ppo = ((ema12 - ema26) / ema26) * 100
    ppo_sig = ppo.ewm(span=9, adjust=False).mean()
    ppo_delta = ppo - ppo_sig

    # Signal-line crosses (histogram zero crosses)
    delta_prev = ppo_delta.shift(1)
    sig_pos_cross = ((ppo_delta > 0) & (delta_prev <= 0)).astype(int)
    sig_neg_cross = ((ppo_delta < 0) & (delta_prev >= 0)).astype(int)

    # Zero-line crosses (raw PPO crosses 0)
    ppo_prev = ppo.shift(1)
    zero_pos_cross = ((ppo > 0) & (ppo_prev <= 0)).astype(int)
    zero_neg_cross = ((ppo < 0) & (ppo_prev >= 0)).astype(int)

    state = np.select(
        [(ppo_delta > 0) & (delta_prev < 0),
         (ppo_delta < 0) & (delta_prev > 0),
         (ppo_delta >= 0) & (delta_prev >= 0),
         (ppo_delta < 0) & (delta_prev < 0)],
        ["Positive CrossOver", "Negative CrossOver", "Trend Positive", "Trend Negative"],
        default="",
    )
    state = pd.Series(state, index=df.index).replace("", np.nan).ffill()
    changed = state != state.shift(1)
    run_id = changed.cumsum()
    trend_count = state.groupby(run_id).cumcount() + 1
    return (ppo, ppo_sig, ppo_delta, trend_count,
            sig_pos_cross, sig_neg_cross, zero_pos_cross, zero_neg_cross)


def _squeeze(df):
    sq = ta.squeeze(high=df["high"], low=df["low"], close=df["close"], **SQZ)
    out = pd.DataFrame(index=df.index)
    if sq is None or "SQZ_ON" not in getattr(sq, "columns", []):
        for c in ["Squeeze_On", "Squeeze_Off", "Squeeze_Fired", "Squeeze_Recent_Fire",
                  "Squeeze_On_Count", "Squeeze_Prior_Length"]:
            out[c] = 0
        out["Squeeze_Mom"] = 0.0
        out["Squeeze_Mom_Pct"] = 0.0
        return out

    on = sq["SQZ_ON"].fillna(0).astype(int)
    off = sq["SQZ_OFF"].fillna(0).astype(int)
    mom_cols = [c for c in sq.columns
                if c.startswith("SQZ_") and c not in ("SQZ_ON", "SQZ_OFF", "SQZ_NO")]
    mom = sq[mom_cols[0]] if mom_cols else pd.Series(0.0, index=df.index)

    out["Squeeze_On"] = on
    out["Squeeze_Off"] = off
    out["Squeeze_Mom"] = mom
    out["Squeeze_Mom_Pct"] = (mom / df["close"].replace(0, np.nan)).fillna(0.0) * 100

    fired = ((on.shift(1) == 1) & (off == 1)).astype(int)
    out["Squeeze_Fired"] = fired
    out["Squeeze_Recent_Fire"] = (
        (fired.rolling(SQZ_RECENT_WINDOW, min_periods=1).max() > 0) & (on == 0)
    ).astype(int)

    run = (on != on.shift(1)).cumsum()
    out["Squeeze_On_Count"] = on.groupby(run).cumsum()
    run_len = on.groupby(run).transform("size")
    last_on_len = (run_len * on).replace(0, np.nan).ffill().fillna(0)
    out["Squeeze_Prior_Length"] = np.where(
        on == 1, out["Squeeze_On_Count"], last_on_len
    ).astype(int)
    return out


def compute_component_frame(df):
    close = df["close"]
    adx = df.ta.adx()
    rsi = df.ta.rsi()
    (ppo, ppo_sig, ppo_delta, trend_count,
     sig_pos_cross, sig_neg_cross, zero_pos_cross, zero_neg_cross) = _trend_count_and_cross(df)
    sqz = _squeeze(df)

    ema20 = ta.ema(close, length=EMA_FAST)
    ema50 = ta.ema(close, length=EMA_MID)
    ema200 = ta.ema(close, length=EMA_SLOW)

    frame = pd.DataFrame(index=df.index)
    frame["Momentum_12_1"] = close.pct_change(MOM_LOOKBACK).shift(MOM_SKIP)
    frame["ADX_Trend"] = adx["ADX_14"]
    frame["High_52W_Ratio"] = close / close.rolling(HIGH_WINDOW, min_periods=HIGH_MINP).max()
    frame["PPO"] = ppo
    frame["PPO_Signal"] = ppo_sig
    frame["PPO_Delta"] = ppo_delta
    frame["DMI_Value"] = adx["DMP_14"] - adx["DMN_14"]
    frame["OBV_slope"] = _obv_slope(df)
    frame["Trend_Count"] = trend_count.astype(float)
    frame["RSI_14"] = rsi
    frame = frame.join(sqz)

    frame["EMA20"] = ema20
    frame["EMA50"] = ema50
    frame["EMA200"] = ema200
    frame["Close_vs_EMA20"] = (close / ema20 - 1.0) * 100
    frame["Close_vs_EMA50"] = (close / ema50 - 1.0) * 100
    frame["Close_vs_EMA200"] = (close / ema200 - 1.0) * 100
    frame["Below_EMA20"] = (close < ema20).astype(int)
    frame["Below_EMA50"] = (close < ema50).astype(int)
    frame["Below_EMA200"] = (close < ema200).astype(int)

    frame["PPO_Sig_Pos_Cross"] = sig_pos_cross
    frame["PPO_Sig_Neg_Cross"] = sig_neg_cross
    frame["PPO_Zero_Pos_Cross"] = zero_pos_cross
    frame["PPO_Zero_Neg_Cross"] = zero_neg_cross
    frame["PPO_Above_Signal"] = (ppo_delta > 0).astype(int)
    frame["PPO_Above_Zero"] = (ppo > 0).astype(int)
    frame["PPO_Pos_Cross"] = sig_pos_cross
    frame["PPO_Neg_Cross"] = sig_neg_cross
    frame["PPO_Negative"] = (ppo_delta < 0).astype(int)

    frame["Last_Close"] = close
    frame["Dollar_Volume_20d"] = (close * df["volume"]).rolling(LIQ_WINDOW, min_periods=15).median()
    return frame


def compute_ticker(df):
    df = df.tail(TRIM_BARS).copy()
    frame = compute_component_frame(df)
    row = frame.iloc[-1].to_dict()

    if pd.isna(row.get("Momentum_12_1")):
        row["Momentum_12_1"] = 0.0
    if pd.isna(row.get("High_52W_Ratio")):
        row["High_52W_Ratio"] = 1.0
    for k, v in list(row.items()):
        if pd.isna(v):
            row[k] = 0.0

    row["N_Bars"] = len(df)
    row["Momentum_Data_OK"] = int(len(df) >= MIN_BARS)

    # ----- Sell-signal flags (latest bar) -----
    row["RSI_Overbought"] = int(row["RSI_14"] >= RSI_OVERBOUGHT)
    row["RSI_Extreme"] = int(row["RSI_14"] >= RSI_EXTREME)
    row["Sell_PPO_Sig_Neg_Cross"] = int(row.get("PPO_Sig_Neg_Cross", 0) == 1)
    row["Sell_PPO_Sig_Pos_Cross"] = int(row.get("PPO_Sig_Pos_Cross", 0) == 1)
    row["Sell_PPO_Zero_Neg_Cross"] = int(row.get("PPO_Zero_Neg_Cross", 0) == 1)
    row["Sell_PPO_Zero_Pos_Cross"] = int(row.get("PPO_Zero_Pos_Cross", 0) == 1)
    row["Sell_PPO_Below_Signal"] = int(row.get("PPO_Negative", 0) == 1)
    row["Sell_PPO_Below_Zero"] = int(row.get("PPO_Above_Zero", 1) == 0)
    row["Sell_PPO_Neg_Cross"] = row["Sell_PPO_Sig_Neg_Cross"]
    row["Sell_PPO_Negative"] = row["Sell_PPO_Below_Signal"]
    row["Sell_Below_EMA20"] = int(row["Below_EMA20"] == 1)
    row["Sell_Below_EMA50"] = int(row["Below_EMA50"] == 1)
    row["Sell_Below_EMA200"] = int(row["Below_EMA200"] == 1)

    row["Sell_Signal_Score"] = int(min(
        2 * row["Sell_PPO_Sig_Neg_Cross"]
        + 1 * row["Sell_PPO_Zero_Neg_Cross"]
        + 1 * row["RSI_Overbought"]
        + 1 * row["RSI_Extreme"]
        + 1 * row["Sell_Below_EMA20"]
        + 1 * row["Sell_Below_EMA50"]
        + 1 * row["Sell_Below_EMA200"],
        8,
    ))
    row["Sell_Alert"] = int(row["Sell_Signal_Score"] >= SELL_ALERT_THRESHOLD)
    return row


# =============================================================================
#  3.  MOMENTUM COMPOSITES  (ranked vs BENCHMARK history)
# =============================================================================
def _pct100(s):
    return 100 * pd.to_numeric(s, errors="coerce").rank(pct=True, na_option="bottom")


def build_benchmark_distribution(bench_df):
    frame = compute_component_frame(bench_df)
    dist = {}
    for c in BENCH_COMPONENTS:
        vals = pd.to_numeric(frame[c], errors="coerce").dropna().values
        dist[c] = np.sort(vals) if len(vals) else np.array([])
    return dist


def _bench_pct(value, arr):
    if arr is None or len(arr) == 0 or pd.isna(value):
        return np.nan
    return 100.0 * np.mean(arr <= value)


def _blend(df, weights, dist):
    parts, ws = [], []
    for comp, w in weights.items():
        if comp not in df.columns:
            continue
        if dist is None:
            parts.append(_pct100(df[comp]))
        else:
            arr = dist.get(comp)
            if arr is None or len(arr) == 0:
                continue
            parts.append(df[comp].apply(lambda v, a=arr: _bench_pct(v, a)))
        ws.append(w)
    if not parts:
        return pd.Series(np.nan, index=df.index)
    W = np.array(ws, dtype=float)
    W = W / W.sum()
    return pd.Series(np.average(np.column_stack(parts), weights=W, axis=1), index=df.index)


def add_momentum_composites(df, dist, bench_name):
    if dist is None:
        print("  WARNING: no benchmark distribution — falling back to WITHIN-LIST ranks.")
    else:
        print(f"  Momentum composites anchored to: {bench_name} history")

    tb = _blend(df, MOM_TRAILING_WEIGHTS, dist).round(1)
    eb = _blend(df, MOM_EMERGING_WEIGHTS, dist).round(1)
    df["Momentum_Trailing_Pctile"] = tb
    df["Momentum_Emerging_Pctile"] = eb
    df["Momentum_Pctile"] = (MOM_TRAILING_WT * tb + MOM_EMERGING_WT * eb).round(1)
    df["Momentum_Dormancy_Pctile"] = (100 - tb).round(1)
    return df


# =============================================================================
#  4.  SQUEEZE SETUP
# =============================================================================
def add_squeeze_setup(df):
    liq_ok = df["Dollar_Volume_20d"].fillna(0) >= MIN_DOLLAR_VOL
    price_ok = df["Last_Close"].fillna(0) >= MIN_PRICE
    df["Is_Tradeable"] = (liq_ok & price_ok).astype(int)

    df["SQ_Release"] = (df["Squeeze_Recent_Fire"] == 1).astype(int)
    df["SQ_PPO_Turning"] = (df["PPO_Delta"] > 0).astype(int)
    df["SQ_DMI_OK"] = (df["DMI_Value"] > DMI_FLOOR).astype(int)
    df["SQ_Flat_Momentum"] = ((df["Momentum_Pctile"] >= FLAT_MOM_LOW) &
                              (df["Momentum_Pctile"] <= FLAT_MOM_HIGH)).astype(int)
    df["SQ_RSI_Waking"] = ((df["RSI_14"] >= RSI_LOW) &
                           (df["RSI_14"] <= RSI_HIGH)).astype(int)
    df["SQ_Room_To_Run"] = (df["High_52W_Ratio"] < ROOM_TO_RUN_MAX).astype(int)
    df["SQ_OBV_Positive"] = (df["OBV_slope"] > 0).astype(int)
    df["SQ_Real_Base"] = (df["Squeeze_Prior_Length"] >= MIN_COMPRESSION).astype(int)
    df["SQ_Data_OK"] = (df["Momentum_Data_OK"] == 1).astype(int)
    df["SQ_Liquid"] = (df["Is_Tradeable"] == 1).astype(int)

    df["Squeeze_Setup_Strength"] = (
        df["SQ_Release"] + df["SQ_PPO_Turning"] + df["SQ_DMI_OK"] +
        df["SQ_Flat_Momentum"] + df["SQ_RSI_Waking"] + df["SQ_Room_To_Run"] +
        df["SQ_OBV_Positive"] + df["SQ_Real_Base"]
    ).astype(int)

    df["Squeeze_Setup_Flag"] = (
        (df["SQ_Release"] == 1) & (df["SQ_PPO_Turning"] == 1) &
        (df["SQ_DMI_OK"] == 1) & (df["SQ_Flat_Momentum"] == 1) &
        (df["SQ_RSI_Waking"] == 1) & (df["SQ_Room_To_Run"] == 1) &
        (df["SQ_Data_OK"] == 1) & (df["SQ_Real_Base"] == 1) &
        (df["SQ_Liquid"] == 1)
    ).astype(int)
    return df


# =============================================================================
#  5.  MAIN
# =============================================================================
def build(tickers=TICKERS, period=PERIOD, benchmark=BENCHMARK, bench_period=BENCH_PERIOD):
    print(f"Fetching {len(tickers)} tickers from Yahoo Finance ({period}) ...")
    prices = fetch_prices(tickers, period)
    if not prices:
        raise SystemExit("No usable price data returned.")

    rows = {}
    for t, df in prices.items():
        try:
            rows[t] = compute_ticker(df)
        except Exception as e:
            print(f"  WARNING: indicator calc failed for {t} ({e}) — skipped")
    out = pd.DataFrame.from_dict(rows, orient="index")
    out.index.name = "Ticker"

    print(f"Fetching benchmark {benchmark} ({bench_period}) for the composite anchor ...")
    dist = None
    try:
        bench_df = fetch_one(benchmark, bench_period)
        if len(bench_df) >= MIN_BARS + 30:
            dist = build_benchmark_distribution(bench_df)
        else:
            print(f"  WARNING: {benchmark} returned only {len(bench_df)} bars.")
    except Exception as e:
        print(f"  WARNING: benchmark fetch failed ({e}).")

    out = add_momentum_composites(out, dist, benchmark)
    out = add_squeeze_setup(out)

    seven = ["Momentum_12_1", "ADX_Trend", "High_52W_Ratio",
             "PPO_Delta", "DMI_Value", "OBV_slope", "Trend_Count"]
    composites = ["Momentum_Trailing_Pctile", "Momentum_Emerging_Pctile",
                  "Momentum_Pctile", "Momentum_Dormancy_Pctile"]
    squeeze = ["Squeeze_On", "Squeeze_Off", "Squeeze_Fired", "Squeeze_Recent_Fire",
               "Squeeze_On_Count", "Squeeze_Prior_Length", "Squeeze_Mom", "Squeeze_Mom_Pct"]
    setup_components = ["SQ_Release", "SQ_PPO_Turning", "SQ_DMI_OK", "SQ_Flat_Momentum",
                        "SQ_RSI_Waking", "SQ_Room_To_Run", "SQ_OBV_Positive",
                        "SQ_Real_Base", "SQ_Data_OK", "SQ_Liquid"]
    setup = ["Squeeze_Setup_Strength", "Squeeze_Setup_Flag"]
    sell_raw = [
        "EMA20", "EMA50", "EMA200",
        "Close_vs_EMA20", "Close_vs_EMA50", "Close_vs_EMA200",
        "Below_EMA20", "Below_EMA50", "Below_EMA200",
        "PPO", "PPO_Signal", "PPO_Delta",
        "PPO_Sig_Pos_Cross", "PPO_Sig_Neg_Cross",
        "PPO_Zero_Pos_Cross", "PPO_Zero_Neg_Cross",
        "PPO_Above_Signal", "PPO_Above_Zero",
        "PPO_Pos_Cross", "PPO_Neg_Cross", "PPO_Negative",
    ]
    sell_flags = [
        "RSI_Overbought", "RSI_Extreme",
        "Sell_PPO_Sig_Neg_Cross", "Sell_PPO_Sig_Pos_Cross",
        "Sell_PPO_Zero_Neg_Cross", "Sell_PPO_Zero_Pos_Cross",
        "Sell_PPO_Below_Signal", "Sell_PPO_Below_Zero",
        "Sell_PPO_Neg_Cross", "Sell_PPO_Negative",
        "Sell_Below_EMA20", "Sell_Below_EMA50", "Sell_Below_EMA200",
        "Sell_Signal_Score", "Sell_Alert",
    ]
    context = ["RSI_14", "Is_Tradeable", "Momentum_Data_OK", "N_Bars",
               "Last_Close", "Dollar_Volume_20d"]

    keep = seven + composites + squeeze + setup_components + setup + sell_raw + sell_flags + context
    seen, ordered = set(), []
    for c in keep:
        if c in out.columns and c not in seen:
            ordered.append(c)
            seen.add(c)
    out = out[ordered]

    out.to_csv(OUTPUT_CSV)
    print(f"\nSaved full table -> {OUTPUT_CSV}")

    print("\n" + "=" * 70)
    print("SELL-SIGNAL SUMMARY (current holdings monitor)")
    print("=" * 70)
    alerts = out[out["Sell_Alert"] == 1].sort_values("Sell_Signal_Score", ascending=False)
    if len(alerts):
        cols = ["Sell_Signal_Score", "RSI_14", "Sell_PPO_Neg_Cross",
                "Sell_PPO_Zero_Neg_Cross", "Sell_Below_EMA20",
                "Sell_Below_EMA50", "Sell_Below_EMA200",
                "Close_vs_EMA20", "Last_Close"]
        print(alerts[[c for c in cols if c in alerts.columns]].round(2).to_string())
    else:
        print("No names currently trigger Sell_Alert (score ≥ 3).")
    print(f"\nNames with RSI ≥ 70: {(out['RSI_Overbought']==1).sum()}")
    print(f"Names with PPO Signal Neg Cross: {(out['Sell_PPO_Sig_Neg_Cross']==1).sum()}")
    print(f"Names with PPO Zero Neg Cross: {(out['Sell_PPO_Zero_Neg_Cross']==1).sum()}")
    print(f"Names below EMA20: {(out['Sell_Below_EMA20']==1).sum()}")
    return out

if __name__ == "__main__":
    output_df = build()
    output_df = output_df.reset_index()
   

Fetching 50 tickers from Yahoo Finance (2y) ...
Fetching benchmark SPY (5y) for the composite anchor ...
  Momentum composites anchored to: SPY history

Saved full table -> G:\My Drive\Projects\Momentum\Momentum.csv

SELL-SIGNAL SUMMARY (current holdings monitor)
        Sell_Signal_Score  RSI_14  Sell_PPO_Neg_Cross  Sell_PPO_Zero_Neg_Cross  Sell_Below_EMA20  Sell_Below_EMA50  Sell_Below_EMA200  Close_vs_EMA20  Last_Close
Ticker                                                                                                                                                           
ACM                     3   34.89                   0                        0                 1                 1                  1           -8.58       62.32
ALGM                    3   37.77                   0                        0                 1                 1                  1           -8.67       40.54
AMPX                    3   45.96                   0                        0          

In [23]:
#@title Write the HTML File
#!/usr/bin/env python3
"""
Build a self-contained HTML dashboard from momentum_squeeze_with_sells.csv
No Streamlit required — works directly in Colab via IPython.display.HTML
or by opening the file in any browser.
"""

from pathlib import Path
import json


def load():
    #df = pd.read_csv(CSV_PATH)
    df = output_df.copy()
    df["Ticker"] = df["Ticker"].astype(str)
    for c in df.columns:
        if c != "Ticker":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def fmt(v, kind="num"):
    if pd.isna(v):
        return "—"
    if kind == "pct1":
        return f"{v:.1f}"
    if kind == "pct2":
        return f"{v:+.2f}"
    if kind == "price":
        return f"{v:,.2f}"
    if kind == "int":
        return str(int(v))
    if kind == "vol":
        return f"{v:,.0f}"
    return f"{v:.4f}"


def score_class(score):
    if score >= 4:
        return "bad"
    if score >= 3:
        return "warn"
    if score >= 2:
        return "mild"
    return ""


def rsi_class(rsi):
    if rsi >= 80:
        return "bad"
    if rsi >= 70:
        return "warn"
    if rsi <= 30:
        return "good"
    return ""


def pctile_class(p):
    if p >= 70:
        return "good"
    if p >= 55:
        return "ok"
    if p <= 30:
        return "bad"
    if p <= 45:
        return "mild"
    return ""


def build_html(df: pd.DataFrame) -> str:
    n = len(df)
    n_alert = int((df["Sell_Alert"] == 1).sum())
    n_rsi70 = int((df["RSI_Overbought"] == 1).sum())
    n_ppo_neg = int((df["Sell_PPO_Neg_Cross"] == 1).sum())
    n_below20 = int((df["Sell_Below_EMA20"] == 1).sum())
    avg_score = df["Sell_Signal_Score"].mean()
    avg_mom = df["Momentum_Pctile"].mean()

    sell = df.sort_values(
        ["Sell_Alert", "Sell_Signal_Score", "RSI_14"],
        ascending=[False, False, False],
    )

    sell_rows = []
    for _, r in sell.iterrows():
        sig_neg = r.get("Sell_PPO_Sig_Neg_Cross", r.get("Sell_PPO_Neg_Cross", 0))
        zero_neg = r.get("Sell_PPO_Zero_Neg_Cross", 0)
        sell_rows.append(f"""
        <tr class="{'alert-row' if r.Sell_Alert == 1 else ''}">
          <td><b>{r.Ticker}</b></td>
          <td class="{score_class(r.Sell_Signal_Score)}">{fmt(r.Sell_Signal_Score, 'int')}</td>
          <td class="{'bad' if r.Sell_Alert else ''}">{'🚨' if r.Sell_Alert else '—'}</td>
          <td class="{rsi_class(r.RSI_14)}">{fmt(r.RSI_14, 'pct1')}</td>
          <td>{fmt(r.PPO, 'pct2')}</td>
          <td>{fmt(r.PPO_Signal, 'pct2')}</td>
          <td class="{'bad' if r.PPO_Delta < 0 else 'good'}">{fmt(r.PPO_Delta, 'pct2')}</td>
          <td class="{'bad' if sig_neg else ''}">{'Yes' if sig_neg else '—'}</td>
          <td class="{'bad' if zero_neg else ''}">{'Yes' if zero_neg else '—'}</td>
          <td class="{'bad' if r.Sell_Below_EMA20 else ''}">{fmt(r.Close_vs_EMA20, 'pct2')}%</td>
          <td class="{'bad' if r.Sell_Below_EMA50 else ''}">{fmt(r.Close_vs_EMA50, 'pct2')}%</td>
          <td class="{'bad' if r.Sell_Below_EMA200 else ''}">{fmt(r.Close_vs_EMA200, 'pct2')}%</td>
          <td class="{pctile_class(r.Momentum_Pctile)}">{fmt(r.Momentum_Pctile, 'pct1')}</td>
          <td>{fmt(r.Last_Close, 'price')}</td>
        </tr>""")

        rank = df.sort_values("Momentum_Pctile", ascending=False)
    rank_rows = []
    for i, (_, r) in enumerate(rank.iterrows(), 1):
        rank_rows.append(f"""
        <tr>
          <td>{i}</td>
          <td><b>{r.Ticker}</b></td>
          <td class="{pctile_class(r.Momentum_Pctile)}">{fmt(r.Momentum_Pctile, 'pct1')}</td>
          <td class="{pctile_class(r.Momentum_Trailing_Pctile)}">{fmt(r.Momentum_Trailing_Pctile, 'pct1')}</td>
          <td class="{pctile_class(r.Momentum_Emerging_Pctile)}">{fmt(r.Momentum_Emerging_Pctile, 'pct1')}</td>
          <td>{fmt(r.Momentum_Dormancy_Pctile, 'pct1')}</td>
          <td class="{score_class(r.Sell_Signal_Score)}">{fmt(r.Sell_Signal_Score, 'int')}</td>
          <td class="{rsi_class(r.RSI_14)}">{fmt(r.RSI_14, 'pct1')}</td>
          <td>{fmt(r.Squeeze_Setup_Strength, 'int')}</td>
          <td>{fmt(r.Last_Close, 'price')}</td>
        </tr>""")

    sqz = df.sort_values(
        ["Squeeze_Setup_Flag", "Squeeze_Setup_Strength"],
        ascending=[False, False],
    )
    sqz_rows = []
    for _, r in sqz.iterrows():
        flags = "".join(
            f'<span class="pill {"on" if r[c] else "off"}">{c.replace("SQ_", "")}</span>'
            for c in [
                "SQ_Release", "SQ_PPO_Turning", "SQ_DMI_OK", "SQ_Flat_Momentum",
                "SQ_RSI_Waking", "SQ_Room_To_Run", "SQ_OBV_Positive", "SQ_Real_Base",
            ]
        )
        sqz_rows.append(f"""
        <tr>
          <td><b>{r.Ticker}</b></td>
          <td class="{'good' if r.Squeeze_Setup_Strength >= 6 else ''}">{fmt(r.Squeeze_Setup_Strength, 'int')}</td>
          <td>{'✅' if r.Squeeze_Setup_Flag else '—'}</td>
          <td class="pills">{flags}</td>
          <td class="{pctile_class(r.Momentum_Pctile)}">{fmt(r.Momentum_Pctile, 'pct1')}</td>
          <td class="{score_class(r.Sell_Signal_Score)}">{fmt(r.Sell_Signal_Score, 'int')}</td>
          <td>{fmt(r.Last_Close, 'price')}</td>
        </tr>""")

    scatter_data = {
        "x": df["Momentum_Trailing_Pctile"].tolist(),
        "y": df["Momentum_Emerging_Pctile"].tolist(),
        "text": df["Ticker"].tolist(),
        "size": (df["Sell_Signal_Score"].fillna(0) + 1).tolist(),
        "color": df["Momentum_Pctile"].tolist(),
        "rsi": df["RSI_14"].tolist(),
        "sell": df["Sell_Signal_Score"].tolist(),
        "close": df["Last_Close"].tolist(),
        "ema20": df["Close_vs_EMA20"].tolist(),
    }

    rsi_vals = df["RSI_14"].dropna().tolist()
    ema20_df = df.sort_values("Close_vs_EMA20")
    ema_bar = {
        "tickers": ema20_df["Ticker"].tolist(),
        "vals": ema20_df["Close_vs_EMA20"].tolist(),
    }

    alerts_list = sell[sell["Sell_Alert"] == 1]
    if len(alerts_list):
        alert_html = "<div class='banner bad'>🚨 ACTIVE SELL ALERTS: " + ", ".join(
            f"<b>{r.Ticker}</b> (score {int(r.Sell_Signal_Score)}, RSI {r.RSI_14:.0f})"
            for _, r in alerts_list.iterrows()
        ) + "</div>"
    else:
        alert_html = "<div class='banner good'>✅ No names currently trigger Sell_Alert (Score ≥ 3)</div>"

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>Momentum · Squeeze · Sell Monitor</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
  :root {{
    --bg: #0d1117; --card: #161b22; --border: #30363d;
    --text: #e6edf3; --muted: #8b949e;
    --good: #1a7f37; --ok: #2ea043; --warn: #d29922; --bad: #cf222e; --mild: #9e6a03;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif;
    background: var(--bg); color: var(--text); line-height: 1.45;
  }}
  header {{
    padding: 1.25rem 1.5rem; border-bottom: 1px solid var(--border);
    background: var(--card); position: sticky; top: 0; z-index: 10;
  }}
  header h1 {{ margin: 0 0 .25rem; font-size: 1.4rem; }}
  header p {{ margin: 0; color: var(--muted); font-size: .9rem; }}
  .kpis {{
    display: grid; grid-template-columns: repeat(auto-fit, minmax(120px, 1fr));
    gap: .75rem; padding: 1rem 1.5rem;
  }}
  .kpi {{
    background: var(--card); border: 1px solid var(--border); border-radius: 8px;
    padding: .75rem 1rem; text-align: center;
  }}
  .kpi .val {{ font-size: 1.5rem; font-weight: 700; }}
  .kpi .lbl {{ font-size: .75rem; color: var(--muted); text-transform: uppercase; }}
  .banner {{
    margin: 0 1.5rem 1rem; padding: .75rem 1rem; border-radius: 8px; font-weight: 600;
  }}
  .banner.bad {{ background: #3d1214; border: 1px solid var(--bad); color: #ff7b72; }}
  .banner.good {{ background: #0d2818; border: 1px solid var(--good); color: #3fb950; }}
  nav {{
    display: flex; gap: .5rem; padding: 0 1.5rem 1rem; flex-wrap: wrap;
  }}
  nav button {{
    background: var(--card); border: 1px solid var(--border); color: var(--text);
    padding: .5rem 1rem; border-radius: 6px; cursor: pointer; font-size: .9rem;
  }}
  nav button.active, nav button:hover {{
    background: #1f6feb; border-color: #1f6feb;
  }}
  section {{
    display: none; padding: 0 1.5rem 2rem;
  }}
  section.active {{ display: block; }}
  h2 {{ font-size: 1.15rem; margin: 0 0 .75rem; }}
  .card {{
    background: var(--card); border: 1px solid var(--border); border-radius: 8px;
    padding: 1rem; margin-bottom: 1rem; overflow-x: auto;
  }}
  table {{
    width: 100%; border-collapse: collapse; font-size: .85rem;
  }}
  th, td {{
    padding: .45rem .6rem; text-align: right; border-bottom: 1px solid var(--border);
    white-space: nowrap;
  }}
  th {{
    text-align: right; color: var(--muted); font-weight: 600;
    position: sticky; top: 0; background: var(--card);
  }}
  th:first-child, td:first-child {{ text-align: left; }}
  tr:hover td {{ background: #1c2128; }}
  tr.alert-row td {{ background: #2d1214; }}
  .good {{ color: #3fb950; font-weight: 600; }}
  .ok {{ color: #56d364; }}
  .warn {{ color: #e3b341; font-weight: 600; }}
  .bad {{ color: #ff7b72; font-weight: 700; }}
  .mild {{ color: #d29922; }}
  .pills {{ text-align: left !important; max-width: 420px; }}
  .pill {{
    display: inline-block; font-size: .65rem; padding: .15rem .35rem;
    border-radius: 4px; margin: 1px; background: #21262d; color: var(--muted);
  }}
  .pill.on {{ background: #1a7f37; color: #fff; }}
  .charts {{ display: grid; grid-template-columns: 1fr 1fr; gap: 1rem; }}
  @media (max-width: 900px) {{ .charts {{ grid-template-columns: 1fr; }} }}
  .note {{ color: var(--muted); font-size: .8rem; margin-top: .5rem; }}
  footer {{
    padding: 1rem 1.5rem; color: var(--muted); font-size: .75rem;
    border-top: 1px solid var(--border);
  }}
</style>
</head>
<body>
<header>
  <h1>📉 Momentum · Squeeze · Holdings Sell Monitor</h1>
  <p>Benchmark-anchored momentum + TTM Squeeze + sell signals (RSI ≥ 70, PPO neg cross, EMA20/50/200)</p>
</header>

<div class="kpis">
  <div class="kpi"><div class="val">{n}</div><div class="lbl">Tickers</div></div>
  <div class="kpi"><div class="val" style="color:#ff7b72">{n_alert}</div><div class="lbl">Sell Alerts</div></div>
  <div class="kpi"><div class="val">{n_rsi70}</div><div class="lbl">RSI ≥ 70</div></div>
  <div class="kpi"><div class="val">{n_ppo_neg}</div><div class="lbl">PPO Neg Cross</div></div>
  <div class="kpi"><div class="val">{n_below20}</div><div class="lbl">Below EMA20</div></div>
  <div class="kpi"><div class="val">{avg_score:.1f}</div><div class="lbl">Avg Sell Score</div></div>
  <div class="kpi"><div class="val">{avg_mom:.0f}</div><div class="lbl">Avg Mom %ile</div></div>
</div>

{alert_html}

<nav>
  <button class="active" onclick="show('sell')">🚨 Sell Signals</button>
  <button onclick="show('rank')">🏆 Momentum Rankings</button>
  <button onclick="show('sqz')">🎯 Squeeze Setups</button>
  <button onclick="show('scatter')">📊 Trailing vs Emerging</button>
  <button onclick="show('guide')">📖 Guide</button>
</nav>

<section id="sell" class="active">
  <h2>Sell Signal / Holdings Monitor</h2>
  <div class="card">
    <table>
      <thead>
        <tr>
          <th>Ticker</th><th>Sell Score</th><th>Alert</th><th>RSI</th>
          <th>PPO</th><th>Signal</th><th>Δ (PPO−Sig)</th>
          <th>Sig Cross↓</th><th>Zero Cross↓</th>
          <th>vs EMA20</th><th>vs EMA50</th><th>vs EMA200</th>
          <th>Mom %ile</th><th>Close</th>
        </tr>
      </thead>
      <tbody>
        {''.join(sell_rows)}
      </tbody>
    </table>
    <p class="note">Sell Score = 2×PPO Signal↓ Cross + 1×PPO Zero↓ Cross + RSI≥70 + RSI≥80 + Below EMA20/50/200. Δ = PPO − Signal (distance). Alert when Score ≥ 3.</p>
  </div>
  <div class="charts">
    <div class="card"><div id="rsi-hist"></div></div>
    <div class="card"><div id="ema-bar"></div></div>
  </div>
</section>

<section id="rank">
  <h2>Momentum Rankings (vs SPY history)</h2>
  <div class="card">
    <table>
      <thead>
        <tr>
          <th>#</th><th>Ticker</th><th>Mom %ile</th><th>Trailing</th><th>Emerging</th>
          <th>Dormancy</th><th>Sell Score</th><th>RSI</th><th>Sqz Str</th><th>Close</th>
        </tr>
      </thead>
      <tbody>
        {''.join(rank_rows)}
      </tbody>
    </table>
  </div>
</section>

<section id="sqz">
  <h2>Squeeze Setup Stack</h2>
  <div class="card">
    <table>
      <thead>
        <tr>
          <th>Ticker</th><th>Strength</th><th>Flag</th><th>Conditions</th>
          <th>Mom %ile</th><th>Sell Score</th><th>Close</th>
        </tr>
      </thead>
      <tbody>
        {''.join(sqz_rows)}
      </tbody>
    </table>
    <p class="note">Green pills = condition true. Strength = count of the 8 core conditions.</p>
  </div>
</section>

<section id="scatter">
  <h2>Trailing vs Emerging Momentum</h2>
  <div class="card"><div id="scatter-plot" style="height:520px"></div></div>
  <p class="note">Bubble size ∝ Sell Score. Color = overall Momentum %ile. Upper-left ≈ dormant→waking; upper-right ≈ strong & accelerating.</p>
</section>

<section id="guide">
  <h2>Interpretation Guide</h2>
  <div class="card">
    <h3>Sell Signal Score (0–7)</h3>
    <ul>
      <li><b>+2</b> PPO crossed <b>below its signal</b> (histogram + → −)</li>
      <li><b>+1</b> Raw PPO crossed <b>below zero</b></li>
      <li><b>+1</b> RSI(14) ≥ 70</li>
      <li><b>+1</b> RSI(14) ≥ 80 (extra)</li>
      <li><b>+1</b> Close &lt; EMA20</li>
      <li><b>+1</b> Close &lt; EMA50</li>
      <li><b>+1</b> Close &lt; EMA200</li>
      <li><b>Δ (PPO−Sig)</b> = distance between PPO and its signal line (shown in table)</li>
    </ul>
    <p><b>Sell_Alert</b> fires when Score ≥ 3.</p>
    <h3>Momentum Composites</h3>
    <p>All percentiles ranked against <b>SPY’s own history</b> (not the peer list).</p>
    <ul>
      <li><b>Trailing</b> = 12-1 return + ADX + proximity to 52w high</li>
      <li><b>Emerging</b> = PPO Δ + DMI + OBV slope + Squeeze Mom + Trend Count</li>
      <li><b>Momentum %ile</b> = 60% Trailing + 40% Emerging</li>
      <li><b>Dormancy</b> = 100 − Trailing</li>
    </ul>
    <h3>Daily workflow</h3>
    <ol>
      <li>Open <b>Sell Signals</b> first — anything with Alert or Score ≥ 3 needs review.</li>
      <li>Fresh PPO Neg Cross = highest urgency.</li>
      <li>RSI ≥ 70 + still above EMAs → consider trim / tighten stops.</li>
      <li>Below EMA20/50 with weak momentum → higher risk of further decline.</li>
    </ol>
  </div>
</section>

<footer>
  Generated from momentum_squeeze_with_sells.csv · Benchmark = SPY ·
  Indicators via yfinance + pandas_ta
</footer>

<script>
const scatter = {json.dumps(scatter_data)};
const rsiVals = {json.dumps(rsi_vals)};
const emaBar = {json.dumps(ema_bar)};

function show(id) {{
  document.querySelectorAll('section').forEach(s => s.classList.remove('active'));
  document.querySelectorAll('nav button').forEach(b => b.classList.remove('active'));
  document.getElementById(id).classList.add('active');
  event.target.classList.add('active');
  if (id === 'scatter') Plotly.Plots.resize('scatter-plot');
  if (id === 'sell') {{
    Plotly.Plots.resize('rsi-hist');
    Plotly.Plots.resize('ema-bar');
  }}
}}

Plotly.newPlot('rsi-hist', [{{
  x: rsiVals, type: 'histogram', nbinsx: 20,
  marker: {{ color: '#58a6ff' }}
}}], {{
  title: 'RSI(14) Distribution',
  paper_bgcolor: '#161b22', plot_bgcolor: '#161b22',
  font: {{ color: '#e6edf3' }},
  xaxis: {{ title: 'RSI', gridcolor: '#30363d' }},
  yaxis: {{ title: 'Count', gridcolor: '#30363d' }},
  shapes: [
    {{ type: 'line', x0: 70, x1: 70, y0: 0, y1: 1, yref: 'paper',
      line: {{ color: '#d29922', dash: 'dash' }} }},
    {{ type: 'line', x0: 80, x1: 80, y0: 0, y1: 1, yref: 'paper',
      line: {{ color: '#cf222e', dash: 'dash' }} }}
  ],
  margin: {{ t: 40, b: 40, l: 40, r: 20 }},
  height: 300
}}, {{responsive: true, displayModeBar: false}});

const emaColors = emaBar.vals.map(v => v >= 0 ? '#3fb950' : '#ff7b72');
Plotly.newPlot('ema-bar', [{{
  x: emaBar.tickers, y: emaBar.vals, type: 'bar',
  marker: {{ color: emaColors }}
}}], {{
  title: 'Distance to EMA20 (%)',
  paper_bgcolor: '#161b22', plot_bgcolor: '#161b22',
  font: {{ color: '#e6edf3', size: 10 }},
  xaxis: {{ tickangle: -45, gridcolor: '#30363d' }},
  yaxis: {{ title: '% vs EMA20', gridcolor: '#30363d', zeroline: true, zerolinecolor: '#8b949e' }},
  margin: {{ t: 40, b: 80, l: 50, r: 20 }},
  height: 300
}}, {{responsive: true, displayModeBar: false}});

Plotly.newPlot('scatter-plot', [{{
  x: scatter.x, y: scatter.y,
  text: scatter.text,
  mode: 'markers+text',
  textposition: 'top center',
  textfont: {{ size: 10, color: '#e6edf3' }},
  marker: {{
    size: scatter.size.map(s => 8 + s * 4),
    color: scatter.color,
    colorscale: 'RdYlGn',
    cmin: 0, cmax: 100,
    colorbar: {{ title: 'Mom %ile', thickness: 12 }},
    line: {{ width: 1, color: '#30363d' }}
  }},
  customdata: scatter.text.map((t,i) => [t, scatter.rsi[i], scatter.sell[i], scatter.close[i], scatter.ema20[i]]),
  hovertemplate: '<b>%{{customdata[0]}}</b><br>Trailing: %{{x:.1f}}<br>Emerging: %{{y:.1f}}<br>RSI: %{{customdata[1]:.1f}}<br>Sell Score: %{{customdata[2]}}<br>Close: %{{customdata[3]:.2f}}<br>vs EMA20: %{{customdata[4]:.2f}}%<extra></extra>'
}}], {{
  paper_bgcolor: '#161b22', plot_bgcolor: '#161b22',
  font: {{ color: '#e6edf3' }},
  xaxis: {{ title: 'Trailing Momentum %ile (vs SPY)', gridcolor: '#30363d', zeroline: false,
            range: [0, 100] }},
  yaxis: {{ title: 'Emerging Momentum %ile (vs SPY)', gridcolor: '#30363d', zeroline: false,
            range: [0, 100] }},
  shapes: [
    {{ type: 'line', x0: 50, x1: 50, y0: 0, y1: 100, line: {{ color: '#484f58', dash: 'dot' }} }},
    {{ type: 'line', x0: 0, x1: 100, y0: 50, y1: 50, line: {{ color: '#484f58', dash: 'dot' }} }}
  ],
  annotations: [
    {{ x: 25, y: 90, text: 'Dormant → Waking', showarrow: false, font: {{ color: '#3fb950', size: 11 }} }},
    {{ x: 75, y: 90, text: 'Strong & Accelerating', showarrow: false, font: {{ color: '#3fb950', size: 11 }} }},
    {{ x: 25, y: 10, text: 'Weak', showarrow: false, font: {{ color: '#ff7b72', size: 11 }} }},
    {{ x: 75, y: 10, text: 'Mature / Cooling?', showarrow: false, font: {{ color: '#d29922', size: 11 }} }}
  ],
  margin: {{ t: 20, b: 50, l: 50, r: 20 }}
}}, {{responsive: true}});
</script>
</body>
</html>
"""
    return html


def main():
    df = load()
    print("DF Loaded")

    html = build_html(df)
    print(f"HTML Built ({len(html):,} bytes)")

    # Write the HTML file
    with open(OUTPUT_HTML, "w", encoding="utf-8", newline="") as f:
        f.write(html)

    # Verify the file was actually written
    #if not OUT_PATH.exists():
    #    raise RuntimeError(f"HTML file was not created: {OUT_PATH}")

    #print(f"Wrote: {OUT_PATH}")
    #print(f"File size: {OUT_PATH.stat().st_size:,} bytes")
    #print(f"Tickers: {len(df)}")
    #print(f"Sell Alerts: {(df.Sell_Alert == 1).sum()}")

    return str(OUTPUT_HTML)


if __name__ == "__main__":
    main()
   


DF Loaded
HTML Built (83,516 bytes)


In [24]:

def finish_up():
    import datetime
    from zoneinfo import ZoneInfo
    print()
    print('Complete - Files Written')
    
    now = datetime.datetime.now(ZoneInfo("America/Denver"))
    # Print the current date and time in the format "Friday, October 6, 2023 at 7:10 AM"
    print(now.strftime("%A, %B %d, %Y at %H:%M %p"))
    print()

finish_up()


Complete - Files Written
Tuesday, August 18, 2026 at 10:51 AM

